In [5]:
import pandas as pd
import numpy as np
import catboost as cb
from sklearn.metrics import mean_absolute_error, r2_score

print("Loading the Parquet masterpieces...")
train_df = pd.read_parquet("../data/model_ready/train_final.parquet")
valid_df = pd.read_parquet("../data/model_ready/valid_final.parquet")
test_df = pd.read_parquet("../data/model_ready/test_final.parquet")

# 1. Split into Features (X) and Target (y)
print("Splitting X and y...")
target_col = 'SalaryNormalized'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=[target_col])
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)
y_test_log = np.log1p(y_test)

print("Waking up the Russian Tank...")


Loading the Parquet masterpieces...
Splitting X and y...
Waking up the Russian Tank...


In [6]:
# CatBoost eats tabular data for breakfast
cat_model = cb.CatBoostRegressor(
    iterations=3000,           # Number of trees
    learning_rate=0.03,        # Slow and steady
    depth=7,                   # Tree depth
    task_type='GPU',           # Force it onto the RTX 4050
    devices='0',               # Use your primary GPU
    od_type='Iter',            # Overfitting detector
    od_wait=200,               # Early stopping (waits 100 rounds)
    random_seed=42,
    verbose=2               # Print updates every 200 trees
)

print("Training CatBoost...")
cat_model.fit(
    X_train, y_train_log,
    eval_set=(X_valid, y_valid_log),
    use_best_model=True
)


Training CatBoost...
0:	learn: 0.4813980	test: 0.3553831	best: 0.3553831 (0)	total: 161ms	remaining: 8m 3s
2:	learn: 0.4622881	test: 0.3377069	best: 0.3377069 (2)	total: 299ms	remaining: 4m 58s
4:	learn: 0.4444789	test: 0.3216618	best: 0.3216618 (4)	total: 362ms	remaining: 3m 36s
6:	learn: 0.4279090	test: 0.3068770	best: 0.3068770 (6)	total: 439ms	remaining: 3m 7s
8:	learn: 0.4126569	test: 0.2935994	best: 0.2935994 (8)	total: 500ms	remaining: 2m 46s
10:	learn: 0.3984600	test: 0.2815593	best: 0.2815593 (10)	total: 564ms	remaining: 2m 33s
12:	learn: 0.3852995	test: 0.2706392	best: 0.2706392 (12)	total: 633ms	remaining: 2m 25s
14:	learn: 0.3731649	test: 0.2609715	best: 0.2609715 (14)	total: 703ms	remaining: 2m 19s
16:	learn: 0.3619228	test: 0.2523044	best: 0.2523044 (16)	total: 761ms	remaining: 2m 13s
18:	learn: 0.3516012	test: 0.2447645	best: 0.2447645 (18)	total: 817ms	remaining: 2m 8s
20:	learn: 0.3420526	test: 0.2381500	best: 0.2381500 (20)	total: 873ms	remaining: 2m 3s
22:	learn: 0.3

CatBoostRegressor(depth=7, devices='0', iterations=3000, learning_rate=0.03, loss_function='RMSE', od_type='Iter', od_wait=200, random_seed=42, task_type='GPU', verbose=2)

In [7]:

print("Predicting...")
y_pred_log = cat_model.predict(X_test)

# Reverse the log cheat code
y_pred_real = np.expm1(y_pred_log)

# Flex the Metrics
mae = mean_absolute_error(y_test, y_pred_real)
r2 = r2_score(y_test, y_pred_real)

print("\n" + "="*30)
print("🚜 CATBOOST RESULTS 🚜")
print("="*30)
print(f"R2 Score: {r2:.4f}")
print(f"MAE:      £{mae:.2f}")
print("="*30 + "\n")

Predicting...

🚜 CATBOOST RESULTS 🚜
R2 Score: 0.6206
MAE:      £5398.43

